Hands-On Lab: Automated Data Enrichment on Danantara Asset Management Data
Executive Summary & Business Context
Danantara Asset Management (Sovereign & Strategic Wealth/Investment Operations) manages multi-billion dollar portfolios, institutional mandates, state-owned enterprise (SOE) allocations, and cross-sector strategic investments.

In this lab, you implement an automated, in-database AI enrichment pipeline connecting Databricks built-in SQL AI functions (ai_classify, ai_extract, ai_summarize, and ai_query) directly to an existing Silver Layer Table (silver_danantara_accounts_and_talent).

We focus on two mission-critical pillars:

Pillar 1: Portfolio Risk & Governance Management:

Detecting counterparty risks, governance red flags, liquidity threats, and regulatory compliance issues from portfolio performance notes and institutional client interactions.

Pillar 2: Talent & Key Person Risk Management:

Detecting key-person attrition risks, burnout signals, competency gaps, and retention urgency among critical investment managers, chief risk officers (CROs), and portfolio analysts.

Step-by-Step Lab Implementation
Step 1: Provision the Silver Baseline Table (Day 1 Integration)
Run the following query in your Databricks SQL Editor or Notebook to create and populate the Silver-tier dataset containing both Portfolio Counterparty / Account Risk and Key Talent Risk records.

In [0]:
CREATE OR REPLACE TABLE training_medallion.`02_silver`.silver_danantara_accounts_and_talent (
  record_id STRING,
  entity_type STRING,            -- 'INSTITUTIONAL_ACCOUNT' or 'KEY_TALENT'
  entity_name STRING,            -- Portfolio/Fund Name or Employee Name
  portfolio_or_dept STRING,      -- e.g., 'Sovereign Energy Transition Fund', 'Private Equity Infra', etc.
  aum_or_compensation DOUBLE,    -- AUM managed (USD) or Annual Compensation
  unstructured_log STRING,       -- Raw audit trail, meeting note, or performance evaluation
  record_timestamp TIMESTAMP
) USING DELTA;

-- Insert realistic Danantara operational logs
INSERT INTO training_medallion.`02_silver`.silver_danantara_accounts_and_talent VALUES
  -- Use Case 1: Portfolio & Compliance Risk
  ('DAN-RSK-001', 'INSTITUTIONAL_ACCOUNT', 'Sovereign Green Infrastructure Fund I', 'Infrastructure & Energy', 1500000000.00,
   'During the Q3 audit, the offshore contractor disclosed a 4-month delay in EPC delivery and unexpected local environmental disputes. State regulatory compliance audit flagged potential covenant breach if milestone 4 is missed by November.', current_timestamp()),

  ('DAN-RSK-002', 'INSTITUTIONAL_ACCOUNT', 'Danantara Strategic Metals Holdings', 'Mining & Strategic Minerals', 850000000.00,
   'Counterparty commodity hedge counterparties requested margin expansion following unexpected nickel and bauxite price volatility. Overall liquidity buffer remains above regulatory threshold, but downside stress test suggests monitoring.', current_timestamp()),

  -- Use Case 2: Talent & Key Person Risk
  ('DAN-TLN-101', 'KEY_TALENT', 'Dr. Faisal Arifin', 'Chief Portfolio Strategist - Renewable Energy', 240000.00,
   'Employee expressed persistent frustration regarding prolonged bureaucratic approval chains for direct private equity investments. Headhunters from a global Singapore-based private credit fund have approached him with an equity partnership offer. Critical key-person retention risk.', current_timestamp()),

  ('DAN-TLN-102', 'KEY_TALENT', 'Dewi Handayani, CFA', 'Senior Quantitative Analyst - Fixed Income', 110000.00,
   'Completed the advanced algorithmic portfolio hedging model 3 weeks ahead of schedule. Requested budget allocation for advanced LLM engineering training and proposed mentorship for junior investment associates.', current_timestamp());

num_affected_rows,num_inserted_rows
4,4


Step 2: Interactive Testing of Individual SQL AI Functions
Test each function individually before orchestrating the full end-to-end pipeline.

1. Severity Classification via ai_classify()
Classify both portfolio risks and talent risks into predefined urgency categories without manual prompt tuning:

In [0]:
SELECT 
  record_id,
  entity_name,
  entity_type,
  ai_classify(
    unstructured_log,
    array('High Risk / Urgent Action Required', 'Moderate Risk / Watchlist', 'Low Risk / Stable Performance')
  ) AS danantara_severity_level
FROM training_medallion.`02_silver`.silver_danantara_accounts_and_talent;

record_id,entity_name,entity_type,danantara_severity_level
DAN-RSK-001,Sovereign Green Infrastructure Fund I,INSTITUTIONAL_ACCOUNT,High Risk / Urgent Action Required
DAN-RSK-002,Danantara Strategic Metals Holdings,INSTITUTIONAL_ACCOUNT,Moderate Risk / Watchlist
DAN-TLN-101,Dr. Faisal Arifin,KEY_TALENT,High Risk / Urgent Action Required
DAN-TLN-102,"Dewi Handayani, CFA",KEY_TALENT,Low Risk / Stable Performance


2. Entity & Threat Extraction via ai_extract()
Extract specific financial or operational risk drivers (e.g., regulatory breaches, competitor poachers, or project delays) directly into structured Spark STRUCTs:

In [0]:
SELECT 
  record_id,
  entity_name,
  ai_extract(
    unstructured_log,
    array('primary_risk_trigger', 'external_counterparty_or_poacher', 'operational_delay_or_breach')
  ) AS extracted_threat_metadata
FROM training_medallion.`02_silver`.silver_danantara_accounts_and_talent
WHERE entity_type = 'INSTITUTIONAL_ACCOUNT';

record_id,entity_name,extracted_threat_metadata
DAN-RSK-001,Sovereign Green Infrastructure Fund I,"List(Q3 audit disclosed offshore contractor 4-month delay in EPC delivery and unexpected local environmental disputes, offshore contractor, 4-month delay in EPC delivery; potential covenant breach if milestone 4 is missed by November)"
DAN-RSK-002,Danantara Strategic Metals Holdings,"List(unexpected nickel and bauxite price volatility, Counterparty commodity hedge counterparties, downside stress test suggests monitoring)"


3. Executive Committee Synopsis via ai_summarize()
Condense dense audit findings into a 15-word executive summary for the Investment Committee (IC) briefing:

In [0]:
SELECT 
  record_id,
  entity_name,
  ai_summarize(unstructured_log, 15) AS ic_executive_summary
FROM training_medallion.`02_silver`.silver_danantara_accounts_and_talent;

record_id,entity_name,ic_executive_summary
DAN-RSK-001,Sovereign Green Infrastructure Fund I,Offshore contractor faces delays and regulatory issues
DAN-RSK-002,Danantara Strategic Metals Holdings,Commodity price volatility affects margin and liquidity
DAN-TLN-101,Dr. Faisal Arifin,"Employee frustrated, offered equity partnership elsewhere"
DAN-TLN-102,"Dewi Handayani, CFA","Completed portfolio model, requested training and mentorship"


Step 3: Production Pipeline – Build the Gold Intelligence Table
Now combine ai_classify(), ai_summarize(), and ai_query() with defensive error handling (failOnError => false) to create the final Gold Layer table:

In [0]:
CREATE OR REPLACE TABLE training_medallion.`03_gold`.gold_danantara_risk_talent_intelligence AS
WITH ai_inference_stage AS (
  SELECT 
    record_id,
    entity_type,
    entity_name,
    portfolio_or_dept,
    aum_or_compensation,
    record_timestamp,
    unstructured_log,

    -- 1. Deterministic Classification
    ai_classify(
      unstructured_log,
      array('Critical Urgent Action', 'Moderate Monitor', 'Low Risk Stable')
    ) AS governance_severity,

    -- 2. Executive Briefing Synthesis
    ai_summarize(unstructured_log, 16) AS executive_brief,

    -- 3. Complex Decision Reasoning with ai_query
    ai_query(
      'system.ai.meta-llama-3-1-8b-instruct',
      CONCAT(
        'Task: Strategic enterprise risk and talent advisory for sovereign asset management. ',
        'Analyze the operational record and output strictly a valid JSON object with keys: ',
        '"action_owner" (e.g., CRO, CHRO, Investment Committee, or Direct Supervisor), ',
        '"recommended_mitigation" (concise recommended step), ',
        '"urgency" (IMMEDIATE, 30_DAYS, or NONE). ',
        'Record: "', unstructured_log, '"'
      ),
      modelParameters => named_struct('temperature', 0.0),
      failOnError => false
    ) AS ai_strategy_struct
  FROM training_medallion.`02_silver`.silver_danantara_accounts_and_talent
),
parsed_stage AS (
  SELECT 
    record_id,
    entity_type,
    entity_name,
    portfolio_or_dept,
    aum_or_compensation,
    record_timestamp,
    governance_severity,
    executive_brief,
    ai_strategy_struct.result AS raw_ai_result,
    ai_strategy_struct.errorMessage AS api_error,
    -- Multi-line regex extraction for resilient JSON extraction
    regexp_extract(ai_strategy_struct.result, '(?s)(\\{.*\\})', 1) AS clean_json
  FROM ai_inference_stage
)
SELECT 
  record_id,
  entity_type,
  entity_name,
  portfolio_or_dept,
  aum_or_compensation,
  governance_severity,
  executive_brief,
  get_json_object(clean_json, '$.action_owner') AS designated_owner,
  get_json_object(clean_json, '$.urgency') AS execution_urgency,
  get_json_object(clean_json, '$.recommended_mitigation') AS strategic_mitigation,
  CASE 
    WHEN api_error IS NOT NULL THEN 'API_ERROR'
    WHEN clean_json = '' OR get_json_object(clean_json, '$.urgency') IS NULL THEN 'UNSTRUCTURED_INSPECTION_REQUIRED'
    ELSE 'PROCESSED'
  END AS pipeline_status,
  record_timestamp
FROM parsed_stage;

num_affected_rows,num_inserted_rows


Step 4: Verification & Gold Intelligence Analytics
Run the following queries to verify the enriched outputs across both use cases.

Verification Query 1: Portfolio Risk Dashboard (For Chief Risk Officer / Investment Committee)

In [0]:
SELECT 
  record_id,
  entity_name AS fund_name,
  portfolio_or_dept,
  aum_or_compensation AS portfolio_aum,
  governance_severity,
  designated_owner,
  execution_urgency,
  strategic_mitigation,
  executive_brief
FROM training_medallion.`03_gold`.gold_danantara_risk_talent_intelligence
WHERE entity_type = 'INSTITUTIONAL_ACCOUNT';

record_id,fund_name,portfolio_or_dept,portfolio_aum,governance_severity,designated_owner,execution_urgency,strategic_mitigation,executive_brief
DAN-RSK-001,Sovereign Green Infrastructure Fund I,Infrastructure & Energy,1.5E9,Critical Urgent Action,Investment Committee,IMMEDIATE,"Review and adjust project timeline to ensure milestone 4 is met by November, and engage local environmental experts to mitigate disputes",Offshore contractor faces delays and regulatory issues
DAN-RSK-002,Danantara Strategic Metals Holdings,Mining & Strategic Minerals,8.5E8,Moderate Monitor,Investment Committee,30_DAYS,Conduct a thorough review of the commodity hedge strategy and consider diversifying the portfolio to mitigate exposure to price volatility,Commodity price volatility affects margin expansion


Expected Outcome: The Green Infrastructure Fund (DAN-RSK-001) is flagged as Critical Urgent Action owned by the Investment Committee or CRO with an IMMEDIATE covenant audit timeline.

Verification Query 2: Key-Person Retention Alert (For Chief Human Resources Officer / Talent Committee)

In [0]:
SELECT 
  record_id,
  entity_name AS employee_name,
  portfolio_or_dept AS role_title,
  governance_severity AS attrition_threat_level,
  designated_owner,
  execution_urgency,
  strategic_mitigation,
  executive_brief
FROM training_medallion.`03_gold`.gold_danantara_risk_talent_intelligence
WHERE entity_type = 'KEY_TALENT';

record_id,employee_name,role_title,attrition_threat_level,designated_owner,execution_urgency,strategic_mitigation,executive_brief
DAN-TLN-101,Dr. Faisal Arifin,Chief Portfolio Strategist - Renewable Energy,Critical Urgent Action,Direct Supervisor,IMMEDIATE,Streamline approval chains for direct private equity investments to reduce bureaucratic delays and increase efficiency,"Employee frustrated, offered equity partnership elsewhere"
DAN-TLN-102,"Dewi Handayani, CFA",Senior Quantitative Analyst - Fixed Income,Low Risk Stable,Investment Committee,30_DAYS,Review and adjust budget allocation for advanced LLM engineering training and mentorship programs to ensure timely completion of future projects.,"Completed portfolio model, requested training budget"


Lab Summary & Key Takeaways for Production
Technique
Benefit in Danantara Production Pipelines

ai_classify() for Sovereign Triage
=>Instant zero-prompt classification of both multi-million dollar covenant breaches and senior talent flight risks.

ai_summarize() for Board Reporting
=>Generates uniform 15-word briefs suitable for Executive Dashboards and Investment Committee digests.

Structured ai_query() + failOnError
=>Assigns clear accountability (action_owner) and mitigation steps without risking pipeline crashes on malformed audit text.

Single-Pass Enrichment to Gold
=>Computes LLM inference once during batch ETL; subsequent BI queries and dashboard refreshes incur zero token cost.